# <span style="color:Thistle">Datos ausentes</span>

#### <span style="color:beige">Sesión 4</span>
---

La diferencia entre los datos que se encuentran en muchos tutoriales y los datos del mundo real es que los datos reales rara vez son limpios y homogéneos. 

En particular, muchos conjuntos de datos interesantes tendrán cierta cantidad de datos ausentes. Para complicar aún más las cosas, cada fuente de datos puede indicar los datos ausentes a su manera.

A partir de este punto,<span style="color:Gold"> nos referiremos a los datos ausentes como *nulos*, *NaN* o *NA*.</span>

## Convención de datos ausentes

Existen varios esquemas que se han desarrollado para indicar la presencia de datos ausentes en una tabla o DataFrame. 

Giran en torno a una de estas dos estrategias: 

### <span style="color:Gold">1. Usar una *máscara* que indique globalmente los valores ausentes</span> 

En el enfoque de máscara, tensemos que asociarlo a la máscara booleana vista en NumPy: un array booleano completamente separado, un bit en la representación de datos que indica el estado nulo de un valor.

<b>Desventaja: </b> Usar un array de máscara separado requiere la asignación de un array booleano adicional, lo que añade sobrecarga tanto en almacenamiento como en cómputo.<br>

### <span style="color:Gold">2. elegir un *valor centinela* que indique una entrada ausente</span> 

En el enfoque centinela, el valor centinela podría ser alguna convención específica de los datos, como indicar un valor entero ausente con -9999 o algún patrón raro, tal y como hicimos con los minijuegos de principio de curso.

O seguir una convención más global, como indicar un valor de punto flotante ausente con NaN (Not a Number), un valor especial que forma parte de la especificación IEEE de punto flotante.

<b>Desventajas: </b> Un valor centinela reduce el rango de valores válidos que se pueden representar y puede requerir lógica adicional (a menudo no optimizada) en la aritmética de CPU y GPU.<br> Los valores especiales comunes como NaN no están disponibles para todos los tipos de datos.<br>

<div class="alert alert-block alert-success">
Como ocurre en la mayoría de los casos en que no existe una elección universalmente óptima, diferentes lenguajes y sistemas utilizan distintas convenciones.<br> Por ejemplo, el lenguaje R usa patrones de bits reservados dentro de cada tipo de dato como valores centinela para indicar datos faltantes.
</div>

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué tipo de dato puede representar un NaN? (int, float, string...)
</div>

## Datos ausentes en Pandas

La forma en que Pandas maneja los valores ausentes está condicionada por su dependencia del paquete NumPy, que sólo tiene integrado los valores NA para tipos de datos que son de punto flotante. Es decir, float.

¿Cual es el motivo de esto?

Pandas podría haber seguido el enfoque de R reservando algún bit para cada tipo de dato individual que indique nulidad, pero este enfoque resulta bastante engorroso. 

Mientras que R contiene cuatro tipos de datos básicos, NumPy soporta *muchos* más: por ejemplo, mientras R tiene un único tipo entero, NumPy soporta *catorce* tipos enteros básicos considerando las precisiones disponibles, el signo y la codificación *endian*. 

Reservar un bit específico en todos los tipos NumPy disponibles llevaría a una cantidad excesiva de casos especiales en diversas operaciones, lo que probablemente necesitaria un fork del paquete NumPy. Además, para los tipos de datos más pequeños (como los enteros de 8 bits), sacrificar un bit para usarlo para indicar que es NaN reduciría significativamente el rango de valores representables.

<span style="color:Gold">Con estas restricciones en mente, Pandas optó por usar centinelas para los datos ausentes, y eligió dos valores nulos de Python ya existentes: el valor especial de punto flotante `NaN` y el objeto `None` de Python. </span>

Esta elección tiene algunos efectos secundarios, como veremos, pero en la práctica resulta ser un buen compromiso en la mayoría de los casos de interés.

### `None`: datos ausentes al estilo Python

El primer valor centinela utilizado por Pandas es `None`, el ya conocido objeto singleton que se usa frecuentemente para representar datos ausentes en código Python. 

Vamos a verlo:

In [1]:
import numpy as np
import pandas as pd

<div class="alert alert-block alert-info">
<b>Pregunta:</b> Antes de ejecutar el código de abajo. ¿Que tipo de variable debería tener el array? (int, float, bool...)
</div>

In [2]:
vals1 = np.array([1, None, 3, 4])
vals1

array([1, None, 3, 4], dtype=object)

Dado que es un objeto Python, `None` no puede usarse en cualquier array NumPy/Pandas arbitrario, sino únicamente en arrays con tipo de dato `'object'` (es decir, arrays de objetos Python).

Este `dtype=object` significa que el mejor tipo que NumPy pudo inferir para el contenido del array son objetos Python. 

Si bien este tipo de array de objetos es útil para algunos propósitos, cualquier operación sobre los datos se realizará a nivel de Python, con una sobrecarga mucho mayor que las operaciones típicamente rápidas que hacemos con arrays con tipos nativos (como int):

In [3]:
for dtype in ['object', 'int']:
    print("tipo =", dtype)
    %timeit np.arange(1E6, dtype=dtype).sum()
    print()

tipo = object
46.9 ms ± 1.64 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

tipo = int
2.58 ms ± 108 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)



El uso de objetos Python en un array también significa que si realizas agregaciones como `sum()` o `min()` sobre un array con un valor `None`, generalmente obtendrás un error:

In [4]:
vals1.sum()

TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'

Esto refleja el hecho de que la suma entre un entero y `None` no está definida.

### `NaN`: datos numéricos ausentes

La otra representación de datos ausentes, `NaN` (acrónimo de *Not a Number*, «No es un Número»), es diferente: es un valor especial de punto flotante reconocido por todos los sistemas que utilizan la representación estándar IEEE de punto flotante (float):

In [5]:
vals2 = np.array([1, np.nan, 3, 4])
vals2.dtype

dtype('float64')

Observa que NumPy eligió un tipo de punto flotante nativo para este array: esto significa que, a diferencia del array de objetos anterior, <span style="color:Gold">este array sí que soporta operaciones rápidas ejecutadas en código compilado y por tanto, eficientes </span>. Es lo que vimos en la unidad anterior.

Debes tener en cuenta que `NaN` es como un virus de datos: infecta cualquier otro objeto con el que entre en contacto.

Es similar al infinito en matemáticas. Independientemente de la operación, el resultado de una aritmética con `NaN` será otro `NaN`:

In [6]:
1 + np.nan

nan

In [7]:
0 * np.nan

nan

Date cuenta que esto significa que las agregaciones sobre los valores no dan un error pero no siempre son útiles:

In [8]:
vals2.sum(), vals2.min(), vals2.max()

(np.float64(nan), np.float64(nan), np.float64(nan))

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Esto se podía evitar con NumPy?
</div>

Ya vimos que NumPy sí proporciona algunas agregaciones especiales que ignorarán estos valores ausentes:

In [9]:
np.nansum(vals2), np.nanmin(vals2), np.nanmax(vals2)

(np.float64(8.0), np.float64(1.0), np.float64(4.0))

Ten en cuenta que `NaN` es específicamente un valor de punto flotante; no existe un valor NaN equivalente para enteros, cadenas u otros tipos.

### NaN y None en Pandas

Ya hemos visto las diferencias entre `NaN` y `None`. Ahora vamos a ver como Pandas está diseñado para manejar los dos de forma casi intercambiable, convirtiendo entre ellos cuando se necesita:

In [10]:
pd.Series([1, np.nan, 2, None])

0    1.0
1    NaN
2    2.0
3    NaN
dtype: float64

Para los tipos que no tienen un valor centinela disponible, Pandas realiza una conversión automática de tipo cuando hay valores inexistentes NA. Por ejemplo, si asignamos `np.nan` a un elemento de un array entero, este será automáticamente convertido a float para que pueda tener un NA:

In [11]:
x = pd.Series(range(2), dtype=int)
x

0    0
1    1
dtype: int64

In [12]:
x[0] = None
x

0    NaN
1    1.0
dtype: float64

Y, además de convertir el array entero a float, Pandas convierte automáticamente `None` a un valor `NaN`.

La siguiente tabla resume las convenciones de promoción de tipo en Pandas cuando se introducen valores NA:

| Clase de tipo | Conversión al almacenar NAs | Valor centinela NA |
|---|---|---|
| `float` (punto flotante) | Sin cambio | `np.nan` |
| `object` (objeto) | Sin cambio | `None` o `np.nan` |
| `integer` (entero) | Convierte a `float64` | `np.nan` |
| `boolean` (booleano) | Convierte a `object` | `None` o `np.nan` |

Ten en cuenta que en Pandas, los datos de tipo cadena siempre se almacenan con dtype `object`.

## Operando con Valores Nulos

Como hemos visto, <span style="color:Gold">Pandas trata `None` y `NaN` como esencialmente intercambiables para indicar valores ausentes o nulos.</span> Para facilitar esta convención, existen varios métodos útiles para detectar, eliminar y reemplazar valores nulos en las estructuras de datos de Pandas:

- `isnull()`: genera una máscara booleana indicando los valores ausentes
- `notnull()`: contrario de `isnull()`
- `dropna()`: devuelve una versión filtrada de los datos
- `fillna()`: devuelve una copia de los datos con los valores ausentes rellenos o imputados

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué significa imputar?
</div>

Vamos a ver estas funciones con ejemplos.

### Máscaras booleana: Detección de valores nulos

Las estructuras de datos de Pandas tienen dos métodos útiles para detectar datos nulos: `isnull()` y `notnull()`. 

Cualquiera de los dos devolverá una máscara booleana sobre los datos. 

Por ejemplo:

In [13]:
data = pd.Series([1, np.nan, 'hola', None])

In [14]:
data.isnull()

0    False
1     True
2    False
3     True
dtype: bool

Las máscaras booleanas pueden usarse directamente como índice de una `Series` o `DataFrame`:

In [15]:
data[data.notnull()]

0       1
2    hola
dtype: object

Los métodos `isnull()` y `notnull()` producen resultados booleanos similares para los `DataFrame`s.

### Eliminación de valores nulos

Además del enmascaramiento, existen los métodos:

- `dropna()` (que <span style="color:Gold">ELIMINA</span> los valores NA)
- `fillna()` (que rellena los valores NA). 

Para una `Series`, el resultado es sencillo:

In [ ]:
data.dropna()

Para un `DataFrame` hay más opciones:

In [ ]:
df = pd.DataFrame([[1,      np.nan, 2],
                   [2,      3,      5],
                   [np.nan, 4,      6]])
df

No podemos eliminar valores individuales de un `DataFrame`; solo podemos eliminar filas o columnas completas. Dependiendo del objetivo, puede que quieras uno u otro, por lo que `dropna()` ofrece varias opciones.

Por defecto, `dropna()` eliminará todas las filas en las que haya *algún* valor nulo:

In [ ]:
df.dropna()

Alternativamente, puedes eliminar los valores NA a lo largo de un eje diferente; `axis=1` elimina todas las columnas que contienen un valor nulo:

In [ ]:
df.dropna(axis='columns')

<span style="color:Gold">Pero esto también elimina datos válidos.</span> Lo cual no tiene mucho sentido en muchos casos, aunque, en función del proyecto puede que sí sea necesario.

Entonces, puede que prefieras eliminar filas o columnas con *todos* los valores NA, o con una mayoría de valores NA. Esto se puede especificar mediante los parámetros `how` o `thresh`, que permiten un control preciso sobre el número de nulos permitidos.

El valor por defecto es `how='any'`, de modo que cualquier fila o columna (dependiendo de la palabra clave `axis`) que contenga un valor nulo será eliminada. 

También puedes especificar `how='all'`, que solo eliminará filas/columnas que sean *todas* valores nulos.

Vamos a ver ejemplos:

In [ ]:
df[3] = np.nan
df

In [ ]:
df.dropna(axis='columns', how='all')

Para un control más fino, el parámetro `thresh` permite especificar un número mínimo de valores no nulos para que la fila/columna se conserve:

In [ ]:
df.dropna(axis='rows', thresh=3)

Aquí se han eliminado la primera y la última fila porque solo contienen dos valores no nulos.

### Relleno de valores nulos

A veces, en lugar de eliminar los valores NA, es preferible reemplazarlos con un valor válido tal y como hicimos en el proyecto de NumPu. 

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Con que rellenamos los valores ausentes en NumPy?
</div>

Este valor podría ser un único número como cero, o podría ser algún tipo de imputación o interpolación a partir de los valores válidos. Se podría hacer usando el método `isnull()` como máscara, pero dado que es una operación tan común, Pandas proporciona el método `fillna()`, que devuelve una copia del array con los valores nulos reemplazados.

Ejemplo:

In [ ]:
data = pd.Series([1, np.nan, 2, None, 3], index=list('abcde'))
data

Podemos rellenar las entradas NA con un único valor, como cero:

In [ ]:
data.fillna(0)

Podemos especificar un relleno hacia adelante (*forward-fill*) para propagar el valor anterior:

In [ ]:
data = pd.Series([1, np.nan, 2, None, 3], index=list('abcde'))

# relleno hacia adelante
data.ffill()

O podemos especificar un relleno hacia atrás (*back-fill*) para propagar los valores siguientes hacia atrás:

In [ ]:
data = pd.Series([1, np.nan, 2, None, 3], index=list('abcde'))
# relleno hacia atrás
data.bfill()

Para los `DataFrame`s, las opciones son similares, pero también podemos especificar un `axis` a lo largo del cual se realiza el relleno:

In [ ]:
df

In [ ]:
df.ffill(axis=1)

Nótese que si no hay un valor anterior disponible durante un relleno hacia adelante, el valor NA permanece.

In [ ]:
df.bfill(axis=1)

In [ ]:
df.ffill(axis=0)